<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex06-training-lab/Ex06_01_loss_functions.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.
- Goodfellow, Bengio & Courville, *Deep Learning*, MIT Press 2016.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_06 · Notebook 01 — Where the Loss Comes From

**Deep Learning for Engineering · Aalborg University · Part 1**

Most courses hand you a list: mean squared error for regression, cross entropy
for classification, and a footnote about Huber. L6.1 does something better. It
gives a **recipe** that generates the loss from an assumption about the noise,
and the list then becomes three worked examples of one idea.

The recipe, in one line:

> **Write down the probability of the data given the model's output. Take the
> logarithm. Put a minus sign in front. That is your loss.**

This notebook makes that concrete. You will

1. derive the mean squared error from a Gaussian noise assumption, and check
   numerically that the two have the **same minimiser**;
2. see what the noise level $\sigma$ does and does not change;
3. derive the cross entropy from a categorical assumption, implement it in
   NumPy, and check it against `nn.CrossEntropyLoss` to machine precision;
4. compare the two losses on the same classification data, and find out that on
   this data the choice matters less than you were told — and exactly where it
   does matter;
5. change the noise assumption to something heavy-tailed and watch a single bad
   reading move a fitted line.

The last one is the point of the whole notebook. **A loss is a statement about
what kind of errors you expect.** Choose the wrong statement and the model does
exactly what you asked, on a question you did not mean to ask.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_6_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex06-training-lab/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

import Ex_6_core as core

core.set_seed(0)

x, y = core.calibration_dataset(n=40)
print("calibration points:", x.shape)
print("true slope %.2f, intercept %.2f, noise sigma %.2f"
      % (core.TRUE_SLOPE, core.TRUE_INTERCEPT, core.TRUE_SIGMA))

fig, ax = plt.subplots(figsize=(6.6, 4.2))
ax.plot(x, core.TRUE_SLOPE * x + core.TRUE_INTERCEPT, lw=1.6, ls="--",
        color="#888888", label="true calibration")
ax.plot(x, y, "o", ms=6, color="#111111", label="readings")
ax.set_xlabel("applied load [normalised]"); ax.set_ylabel("sensor reading")
ax.set_title("Forty calibration points from a load cell")
ax.legend(frameon=False, fontsize=9); ax.grid(alpha=0.25)
plt.show()

**What you should see.** Forty points scattered about a straight dashed line
of slope 2.4 and intercept 0.8.

The story: a load cell is loaded to known values and its reading is recorded. The
instrument is linear, and its error is Gaussian, independent between readings,
and of constant variance. Those three assumptions are exactly the ones the mean
squared error encodes, and in a real calibration you would check all three
against a residual plot rather than assume them.

---

## 1 · From a Gaussian assumption to the mean squared error

> **You have already done the fitting part of this.** In Ex_03 notebook
> 01 you implemented least squares by hand and used it to recover a
> thermal time constant. Nothing below asks you to repeat that. The
> question here is the one Ex_03 could not answer: *why least squares?*
> Of all the ways to score a line against forty points, why that one —
> and when is it the wrong one? The grid search you write is a
> **verification**, not a fit: it checks that two objectives which look
> nothing alike land on the same answer.


Write the model as $\hat{y}(x) = a x + b$ and the assumption as

$$y_i = \hat{y}(x_i) + \varepsilon_i, \qquad
\varepsilon_i \sim \mathcal{N}(0, \sigma^2), \quad \text{independent.}$$

Then the probability density of observing reading $y_i$ is

$$p(y_i \mid a, b) = \frac{1}{\sqrt{2\pi\sigma^2}}
\exp\!\left(-\frac{(y_i - \hat{y}(x_i))^2}{2\sigma^2}\right)$$

and, because the readings are independent, the density of the whole dataset is
the product over $i$. Products of forty small numbers underflow, so take the
logarithm, which turns the product into a sum; and minimise rather than
maximise, so put a minus in front. The **negative log likelihood** is

$$\mathcal{L}_{\mathrm{NLL}}(a, b) \;=\;
\underbrace{\frac{n}{2}\log(2\pi\sigma^2)}_{\text{does not contain } a, b}
\;+\; \frac{1}{2\sigma^2}\sum_{i=1}^{n}\bigl(y_i - \hat{y}(x_i)\bigr)^2 .$$

Look at what that is. The first term does not depend on the parameters at all.
The second is the sum of squared errors, multiplied by a positive constant. So

> **minimising the negative log likelihood of a Gaussian is minimising the mean
> squared error.** Not similar to it — the same problem, with the same minimiser.

MSE is not a reasonable-looking choice somebody made. It is what the Gaussian
assumption forces.

### Your turn

Check it numerically rather than taking it on trust. Evaluate both objectives on
a grid of $(a, b)$ and compare where each is smallest.

In [ ]:
# TODO: write both objectives and find each one's minimiser on a grid.
#
#   def nll(a, b, sigma):
#       residual = y - (a * x + b)
#       n = len(x)
#       return (0.5 * n * np.log(2 * np.pi * sigma ** 2)
#               + np.sum(residual ** 2) / (2 * sigma ** 2))
#
#   def mse(a, b):
#       return np.mean((y - (a * x + b)) ** 2)
#
# Then evaluate both over the grids below, using core.TRUE_SIGMA for sigma
# (the notebook never defines a bare `sigma` variable):
#       a_grid = np.linspace(1.8, 3.0, 601)
#       b_grid = np.linspace(0.2, 1.4, 601)
# building two arrays NLL and MSE of shape (601, 601), and find the (a, b) at
# which each is smallest. np.unravel_index(G.argmin(), G.shape) gives the pair
# of indices.
#
# Record a_nll, b_nll, a_mse, b_mse.

raise NotImplementedError("Evaluate the NLL and the MSE on a grid and locate both minima")

In [ ]:
design = np.stack([x, np.ones_like(x)], axis=1)
a_ls, b_ls = np.linalg.lstsq(design, y, rcond=None)[0]

print(core.error_table(
    [["negative log likelihood (grid)", f"{a_nll:.4f}", f"{b_nll:.4f}"],
     ["mean squared error (grid)", f"{a_mse:.4f}", f"{b_mse:.4f}"],
     ["least squares (exact)", f"{a_ls:.4f}", f"{b_ls:.4f}"],
     ["the truth", f"{core.TRUE_SLOPE:.4f}", f"{core.TRUE_INTERCEPT:.4f}"]],
    ["objective", "slope", "intercept"]))

fig, axes = plt.subplots(1, 2, figsize=(12.4, 4.4))
for ax, G, name in zip(axes, [NLL, MSE],
                       ["negative log likelihood", "mean squared error"]):
    cs = ax.contour(b_grid, a_grid, G, levels=25, cmap="viridis")
    ax.plot(b_ls, a_ls, "x", ms=13, mew=2.4, color="#d94f2b",
            label="least squares")
    ax.plot(core.TRUE_INTERCEPT, core.TRUE_SLOPE, "o", ms=9, mfc="none",
            mec="#111111", mew=2.0, label="truth")
    ax.set_xlabel("intercept $b$"); ax.set_ylabel("slope $a$")
    ax.set_title(name); ax.legend(frameon=False, fontsize=9)
plt.show()

**What you should see.**

| objective | slope | intercept |
| --- | --- | --- |
| negative log likelihood (grid) | 2.3960 | 0.7860 |
| mean squared error (grid) | 2.3960 | 0.7860 |
| least squares (exact) | 2.3954 | 0.7871 |
| the truth | 2.4000 | 0.8000 |

and two contour plots whose ellipses have **identical shapes and identical
centres**, differing only in the numbers on the contour labels.

The first two rows agree exactly, which is the claim. They differ from the third
row in the fourth decimal place, and that difference is the grid spacing —
0.002 in $a$ and $b$ — and nothing else.

The gap to the truth is a different kind of thing entirely. It is **estimation
error**: forty noisy readings do not determine the true calibration, and a
different forty would give a slightly different answer. No loss function fixes
that. Only more data does.

---

## 2 · What $\sigma$ changes, and what it does not

$\sigma$ dropped out of the argument above because it multiplies the sum of
squares by a constant and adds another constant. Constants do not move a
minimiser.

That is worth being precise about, because $\sigma$ is not doing *nothing*.

In [ ]:
print("negative log likelihood at the fitted line, for several assumed sigma:")
for s in (0.1, 0.2, core.TRUE_SIGMA, 0.5, 1.0):
    print(f"   sigma = {s:.2f}   NLL = {nll(a_ls, b_ls, s):9.4f}")

residual = y - (a_ls * x + b_ls)
sigma_hat = float(np.sqrt(np.mean(residual ** 2)))
print(f"\nsigma estimated from the residuals: {sigma_hat:.4f}"
      f"   (the instrument's true value: {core.TRUE_SIGMA:.4f})")

fig, ax = plt.subplots(1, 2, figsize=(12.0, 3.4))
ax[0].plot(x, residual, "o", ms=5, color="#1f77b4")
ax[0].axhline(0.0, color="#111111", lw=1.0)
ax[0].set_xlabel("applied load"); ax[0].set_ylabel("residual")
ax[0].set_title("residuals against load — should be structureless")
ax[1].hist(residual, bins=12, color="#1f77b4", alpha=0.85)
ax[1].set_xlabel("residual"); ax[1].set_title("residual histogram")
for a in ax:
    a.grid(alpha=0.25)
plt.show()

**What you should see.** Negative log likelihoods of about 186.6, 32.9, 14.5,
18.7 and 39.2 for the five values of $\sigma$, an estimated $\sigma$ of about
0.3478 against a true 0.35, and two residual plots showing no obvious structure.

Three things $\sigma$ does, none of which is moving the minimiser.

**It sets the value of the loss**, and therefore whether two different *models*
can be compared by likelihood. Model comparison needs $\sigma$; parameter fitting
does not.

**It can be estimated too.** Minimising the negative log likelihood over
$\sigma$ as well as over $a$ and $b$ gives $\hat{\sigma}^2 = \frac{1}{n}\sum
r_i^2$, the mean squared residual — which is what the printed estimate is. That
it comes out near the instrument's true 0.35 is a check that the model is
adequate. If it came out at 2.0 you would know the model was missing something
real, and no amount of refitting would help.

**It becomes the weight in a multi-term loss.** This is the one that matters for
Part 2. If you have two data sources with different noise levels, the likelihood
gives you

$$\mathcal{L} = \frac{1}{2\sigma_1^2}\sum r_{1,i}^2
+ \frac{1}{2\sigma_2^2}\sum r_{2,j}^2$$

and the relative weight $\sigma_2^2/\sigma_1^2$ is **not a hyperparameter to
tune** — it is the ratio of the variances of your two instruments, which you can
measure. L6.1's fourth bridge slide makes the same point about the $\lambda$ in
$\mathcal{L} = \mathcal{L}_{\mathrm{data}} + \lambda\mathcal{L}_{\mathrm{phys}}$:
it is a statement about how much you trust your model relative to your
instruments.

---

## 3 · From a categorical assumption to the cross entropy

Now change the assumption. The output is not a real number with noise on it; it
is one of $C$ classes. The natural probabilistic statement is that the model
outputs a probability for each class, and the data is a draw from that
distribution.

A network's outputs are unconstrained real numbers — **logits** $z_c$ — so first
turn them into probabilities with the softmax,

$$p_c = \frac{e^{z_c}}{\sum_{k} e^{z_k}},$$

which is positive and sums to one by construction. The probability of the
observed label $t$ is then $p_t$, and the recipe says take minus the log:

$$\mathcal{L} = -\log p_t
= -z_t + \log\sum_k e^{z_k}.$$

Averaged over the dataset, that is the **cross entropy**. Again: not a
reasonable-looking choice, but what the categorical assumption forces.

Two implementation details that are worth more than they look.

**Subtract the maximum before exponentiating.** $e^{z}$ overflows for $z$ above
about 709 in double precision and about 88 in single. Since the softmax is
unchanged by subtracting a constant from every logit, subtract $\max_k z_k$ and
the largest exponent becomes exactly zero. Every production implementation does
this, and it is why the identity above is written with a $\log\sum e$ rather than
as $-\log p_t$ computed in two steps.

**`nn.CrossEntropyLoss` takes logits, not probabilities.** It applies the softmax
internally. Apply your own first and you apply it twice, which trains — slowly,
to a worse answer — and prints no warning.

### Your turn

Implement the softmax and the cross entropy in NumPy, and check against PyTorch.

In [ ]:
# TODO: implement softmax and cross entropy in NumPy, stably.
#
#   def softmax_np(z):                 # z has shape (N, C)
#       shifted = z - z.max(axis=1, keepdims=True)
#       e = np.exp(shifted)
#       return e / e.sum(axis=1, keepdims=True)
#
#   def cross_entropy_np(z, t):        # t has shape (N,), integer labels
#       shifted = z - z.max(axis=1, keepdims=True)
#       log_p   = shifted - np.log(np.exp(shifted).sum(axis=1, keepdims=True))
#       return -log_p[np.arange(len(t)), t].mean()
#
# Then evaluate both on the fixed logits and labels below and compare with
# nn.CrossEntropyLoss. Record ce_numpy and ce_torch.

logits = np.array([[2.0, 1.0, 0.1],
                   [0.5, 2.5, 0.3],
                   [1.2, 0.7, 3.1],
                   [0.0, 0.0, 0.0],
                   [8.0, -2.0, -3.0],
                   [-1.0, -1.0, 9.0]])
labels = np.array([0, 1, 2, 1, 2, 2])

raise NotImplementedError("Implement softmax_np and cross_entropy_np")

In [ ]:
probs = softmax_np(logits)
print("softmax rows sum to one:", np.allclose(probs.sum(axis=1), 1.0))
print("probabilities:")
for row, t in zip(probs, labels):
    print("   " + "  ".join(f"{p:.4f}" for p in row) + f"    label {t}")
print()
print(f"cross entropy, NumPy : {ce_numpy:.10f}")
print(f"cross entropy, torch : {ce_torch:.10f}")
print(f"difference           : {abs(ce_numpy - ce_torch):.2e}")
print()
print("loss for a uniform three-class guess: ln 3 =", round(float(np.log(3)), 6))

**What you should see.** `softmax rows sum to one: True`, a table of
probabilities, and

```
cross entropy, NumPy : 2.1585311972
cross entropy, torch : 2.1585311972
difference           : 0.00e+00
```

or a difference around $10^{-16}$.

Read row four of the probability table: the logits are all zero, so the
probabilities are all one third — a model that has learned nothing. Its
contribution to the loss is $-\log(1/3) = \ln 3 = 1.0986$, which is the number
printed at the bottom. **Memorise it.** A three-class cross entropy that starts
anywhere other than about 1.10 means your labels, your shapes or your
initialisation are wrong, and it is the cheapest sanity check in classification.
The two-class equivalent is $\ln 2 = 0.693$ and the ten-class one is
$\ln 10 = 2.303$.

Row five is the other one to read: logits `[8, -2, -3]` with label 2. The model
is confidently, catastrophically wrong, and its probability for the true class is
about $10^{-5}$, so its contribution to the loss is about 11. The cross entropy
does not merely notice that the model is wrong — it notices *how confidently*, and
that is the subject of the next section.

---

## 4 · The two losses on the same problem

The vibration data has three classes and two features. Train the same small
network twice: once with cross entropy on the logits, and once with mean squared
error on the softmax outputs against one-hot targets.

The second is not a straw man. It is what a great many people write when they
come to classification from a regression background, and it does train.

### Your turn

In [ ]:
# TODO: train the same network under both losses and compare.
#
#   X, y_cls = core.vibration_dataset()
#   X_train, y_train = X[:240], y_cls[:240]
#   X_test,  y_test  = X[240:], y_cls[240:]
#
#   Xa = torch.tensor(X_train); ya = torch.tensor(y_train)
#   Xb = torch.tensor(X_test)
#   Y_onehot = torch.tensor(core.one_hot(y_train).astype(np.float32))
#
#   For each of the two losses:
#       core.set_seed(0)
#       model = nn.Sequential(nn.Linear(2, 16), nn.Tanh(), nn.Linear(16, 3))
#       optimiser = torch.optim.Adam(model.parameters(), lr=0.05)
#       400 epochs; for cross entropy the loss is
#           nn.CrossEntropyLoss()(model(Xa), ya)
#       and for the other it is
#           nn.MSELoss()(torch.softmax(model(Xa), dim=-1), Y_onehot)
#
#   Record, for each, the held-out accuracy and the held-out CROSS ENTROPY --
#   the second is the calibration measure, and both models must be scored on
#   the same yardstick or the comparison means nothing.
#
#   Put them in results = {name: (accuracy, held-out cross entropy)} under the
#   exact keys "cross entropy" and "mean squared error" (the saving cell indexes
#   them by those names), and keep each trained model in models_cls[name] --
#   the decision-boundary plot below uses them.

raise NotImplementedError("Train the same network with cross entropy and with MSE")

In [ ]:
print(core.error_table(
    [[name, f"{acc:.3f}", f"{held:.4f}"] for name, (acc, held) in results.items()],
    ["training loss", "held-out accuracy", "held-out cross entropy"]))

fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.6))
for ax, name in zip(axes, results):
    with torch.no_grad():
        pred = models_cls[name](Xb).numpy().argmax(axis=1)
    core.plot_classes(X_test, y_test, ax=ax, predictions=pred,
                      title=f"trained with {name}")
plt.show()

**What you should see.**

| training loss | held-out accuracy | held-out cross entropy |
| --- | --- | --- |
| cross entropy | 0.967 | 0.1191 |
| mean squared error | 0.967 | 0.1687 |

**The accuracies are identical.** That is the honest result, and it is worth
stating plainly because the textbook version of this experiment usually is not
this close. On a three-class problem with well-separated clusters and 240
training points, both losses find essentially the same decision boundary. If you
were told that using MSE for classification "does not work", this is the
counter-example.

**The held-out cross entropies are not identical.** The model trained with MSE is
about forty per cent worse on the yardstick that measures *probabilities* rather
than *decisions*. It gets the same answers and is less well calibrated about
them — its confidence is a worse estimate of its accuracy. If the downstream use
is "alarm if $p(\text{bearing fault}) > 0.8$", that difference is the whole
system.

So where does the difference come from? Not from the decision boundary. From the
gradient.

---

## 5 · The gradient when the model is confidently wrong

This is the mechanism, and it can be written down exactly. Take the two-class
case, with a single logit $z$ and $p = \sigma(z)$, for a sample whose true label
is 1.

**Cross entropy.** $\mathcal{L} = -\log p$, and after the algebra
$$\frac{\partial \mathcal{L}}{\partial z} = p - 1.$$

**Squared error on the probability.** $\mathcal{L} = (p - 1)^2$, and by the chain
rule, using $\sigma' = p(1-p)$,
$$\frac{\partial \mathcal{L}}{\partial z} = 2(p - 1)\,p\,(1 - p).$$

The second has an extra factor of $p(1-p)$, which goes to zero at both ends. Put
numbers in it.

In [ ]:
print("  logit z    p        dCE/dz      dMSE/dz")
for z in (-6.0, -3.0, 0.0, 3.0, 6.0):
    p = 1.0 / (1.0 + np.exp(-z))
    g_ce = p - 1.0
    g_mse = 2.0 * (p - 1.0) * p * (1.0 - p)
    print(f"  {z:6.1f}   {p:.4f}   {g_ce: .4f}   {g_mse: .6f}")

zs = np.linspace(-8, 8, 400)
ps = 1.0 / (1.0 + np.exp(-zs))
fig, ax = plt.subplots(figsize=(6.8, 4.2))
ax.plot(zs, np.abs(ps - 1.0), lw=1.9, color="#d94f2b", label="cross entropy")
ax.plot(zs, np.abs(2 * (ps - 1) * ps * (1 - ps)), lw=1.9, color="#1f77b4",
        label="squared error on the probability")
ax.set_yscale("log")
ax.set_xlabel("logit $z$   (true label is 1, so $z \\ll 0$ is confidently wrong)")
ax.set_ylabel("|gradient with respect to $z$|")
ax.set_title("What each loss does about a confident mistake")
ax.legend(frameon=False, fontsize=9); ax.grid(alpha=0.25, which="both")
plt.show()

**What you should see.**

```
  logit z    p        dCE/dz      dMSE/dz
    -6.0   0.0025   -0.9975   -0.004921
    -3.0   0.0474   -0.9526   -0.086068
     0.0   0.5000   -0.5000   -0.250000
     3.0   0.9526   -0.0474   -0.004285
     6.0   0.9975   -0.0025   -0.000012
```

and a log-scale figure in which the red curve flattens on the left while the blue
one collapses.

Read the first row. The model is as wrong as it can be — it assigns probability
0.0025 to the correct class. Cross entropy responds with a gradient of $-0.9975$,
essentially the maximum it can produce. Squared error responds with $-0.0049$:
**two hundred times smaller**. The sample the model most needs to learn from is
the sample it learns least from.

That is the reason cross entropy is the standard choice, and it is a statement
about optimisation, not about the location of the optimum. On the vibration data,
where no sample is ever confidently wrong for long, the two agree; on a hard
problem with mislabelled or genuinely difficult examples, they do not.

Notice also the symmetry of the squared-error curve. It is small on the right as
well — for samples that are confidently **right**, which is correct behaviour, and
cross entropy does the same thing there. The two losses differ only on the left,
and only where it matters.

---

## 6 · Change the noise assumption, change the loss

The last experiment closes the loop. If MSE follows from a Gaussian assumption,
then MSE is the wrong loss whenever the noise is not Gaussian — and the most
common way for that to be true in engineering is a **bad reading**: a stuck bit,
a knocked cable, a transcription error.

A Gaussian says a residual of $10\sigma$ has probability about $10^{-23}$. The
squared error therefore treats one such point as worth a hundred points at
$\sigma$, and moves the whole line to accommodate it. Assume a heavier-tailed
distribution instead — the Laplace, $p(r) \propto e^{-|r|/b}$ — and the recipe
gives $-\log p \propto |r|$: the **mean absolute error**, which weights a large
residual only linearly.

### Your turn

Corrupt one reading and fit the same line under both losses.

In [ ]:
# TODO: fit a straight line to the corrupted data under MSE and under MAE.
#
#   x_bad, y_bad = core.add_outlier(x, y, index=7, offset=4.0)
#
#   For each loss in (nn.MSELoss(), nn.L1Loss()):
#       core.set_seed(0)
#       line = nn.Linear(1, 1)
#       optimiser = torch.optim.Adam(line.parameters(), lr=0.05)
#       3000 epochs on core.to_tensor(x_bad) and core.to_tensor(y_bad)
#       record (float(line.weight.item()), float(line.bias.item()))
#
#   Put them in fits = {"MSE": (slope, intercept), "MAE": (slope, intercept)}.
#
# nn.L1Loss is the mean absolute error. 3000 epochs is generous; the MAE
# objective has a kink at every data point and converges more slowly.

raise NotImplementedError("Fit the corrupted data under both losses")

In [ ]:
print(core.error_table(
    [[name, f"{a:.4f}", f"{b:.4f}",
      f"{abs(a - core.TRUE_SLOPE):.4f}", f"{abs(b - core.TRUE_INTERCEPT):.4f}"]
     for name, (a, b) in fits.items()]
    + [["truth", f"{core.TRUE_SLOPE:.4f}", f"{core.TRUE_INTERCEPT:.4f}",
        "-", "-"]],
    ["loss", "slope", "intercept", "slope error", "intercept error"]))

grid = np.linspace(0, 2, 100)
fig, ax = plt.subplots(figsize=(7.0, 4.4))
ax.plot(grid, core.TRUE_SLOPE * grid + core.TRUE_INTERCEPT, lw=1.6, ls="--",
        color="#888888", label="truth")
for i, (name, (a, b)) in enumerate(fits.items()):
    ax.plot(grid, a * grid + b, lw=2.0,
            color=["#d94f2b", "#1f77b4"][i], label=f"fitted with {name}")
ax.plot(x_bad, y_bad, "o", ms=6, color="#111111", label="readings")
ax.plot(x_bad[7], y_bad[7], "o", ms=13, mfc="none", mec="#d94f2b", mew=2.4,
        label="the bad reading")
ax.set_xlabel("applied load"); ax.set_ylabel("sensor reading")
ax.set_title("One bad reading in forty")
ax.legend(frameon=False, fontsize=9); ax.grid(alpha=0.25)
plt.show()

**What you should see.**

| loss | slope | intercept | slope error | intercept error |
| --- | --- | --- | --- | --- |
| MSE | 2.1824 | 1.0762 | 0.2176 | 0.2762 |
| MAE | 2.3616 | 0.8086 | 0.0384 | 0.0086 |
| truth | 2.4000 | 0.8000 | - | - |

and a figure in which the red MSE line is visibly dragged towards the circled
point while the blue MAE line ignores it.

One corrupted reading in forty moved the least-squares slope by nine per cent and
the intercept by thirty-five per cent. The absolute-error fit is within two per
cent of the truth on both — essentially the same answer it would have given
without the corruption.

The lesson is not "always use MAE". MAE has real costs: it is not differentiable
at zero, it converges more slowly, and if the noise really is Gaussian it is a
less efficient estimator — it throws away information you paid for. The lesson is
the one the whole notebook has been building:

> **The loss encodes an assumption about the noise. If you cannot say what
> distribution your loss corresponds to, you do not know what you assumed.**

The practical middle ground is the **Huber** loss, which is quadratic for small
residuals and linear for large ones — Gaussian in the middle, Laplace in the
tails. It is available as `nn.SmoothL1Loss`, and it is what you would actually
reach for on instrument data. Notebook 05 suggests adding it to this comparison.

---

## 7 · Save

In [ ]:
os.makedirs(core.OUTPUT_DIR, exist_ok=True)
path = os.path.join(core.OUTPUT_DIR, "nb01_losses.npz")
np.savez(path,
         a_nll=a_nll, b_nll=b_nll, a_mse=a_mse, b_mse=b_mse,
         a_ls=a_ls, b_ls=b_ls, sigma_hat=sigma_hat,
         ce_numpy=ce_numpy, ce_torch=ce_torch,
         acc_ce=results["cross entropy"][0],
         acc_mse=results["mean squared error"][0],
         heldout_ce_ce=results["cross entropy"][1],
         heldout_ce_mse=results["mean squared error"][1],
         fit_mse=np.asarray(fits["MSE"]), fit_mae=np.asarray(fits["MAE"]))
print("wrote", path)

**What you should see.** `wrote .../Ex06_outputs/nb01_losses.npz`.

---

## 8 · Before you move on

Answer these here.

1. The negative log likelihood and the mean squared error had the same
   minimiser. Give one thing you can do with the first that you cannot do with
   the second.
2. Your fitted $\hat{\sigma}$ came out near the instrument's true value. Suppose
   it had come out four times larger. Name two different explanations and say how
   you would tell them apart.
3. Both losses gave the same accuracy on the vibration data but different
   held-out cross entropies. Describe an application in which you would not care,
   and one in which you would refuse to deploy the worse-calibrated model.
4. You are given a dataset in which about one reading in fifty is a
   transcription error, and the rest are Gaussian. Write down the loss you would
   use and the assumption it corresponds to.

---

Continue with **`Ex06_02_optimiser_comparison.ipynb`**, which takes the loss as
given and asks how to get to the bottom of it.

*Write your answers here.*

1.
2.
3.
4.

---

Next: **notebook 02**, where this same loss is minimised four different
ways — and where the Adam-then-L-BFGS recipe that every exercise in Part 2
uses is measured rather than asserted.
